# factlens — Geometric Hallucination Detection

**Detect LLM hallucinations using embedding geometry. No second LLM. Deterministic. Auditable.**

This notebook demonstrates factlens in under 5 minutes. Everything runs in the browser — no API keys needed.

factlens uses two geometric scores:

| Score | When to use | What it measures |
|-------|-------------|------------------|
| **SGI** (Semantic Grounding Index) | You have source context | Is the response closer to the context or the question? |
| **DGI** (Directional Grounding Index) | No context available | Does the question→response direction match known grounded patterns? |

📄 Papers: [SGI (arXiv:2512.13771)](https://arxiv.org/abs/2512.13771) · [DGI (arXiv:2602.13224)](https://arxiv.org/abs/2602.13224)  
📦 Repo: [github.com/factlens/factlens](https://github.com/factlens/factlens)  
📚 Docs: [docs.factlens.dev](https://docs.factlens.dev)

## 1. Install

In [ ]:
!pip install -q factlens

## 2. SGI — Context-Based Verification

When you have source context (e.g., RAG retrieval), SGI asks: **is the response geometrically closer to the context or to the question?**

$$\text{SGI} = \frac{\|\phi(r) - \phi(q)\|}{\|\phi(r) - \phi(\text{ctx})\|}$$

- SGI > 1.2 → strong context engagement (pass)
- SGI < 0.95 → weak engagement (flagged for review)

Let's test it with a grounded response and a hallucinated one.

In [ ]:
from factlens import compute_sgi

question = "How does compound interest work on a savings account?"

context = (
    "Compound interest calculates interest on both the initial principal "
    "and the accumulated interest from previous periods. This causes the "
    "balance to grow faster over time compared to simple interest, which "
    "only calculates interest on the principal amount."
)

# A grounded response — uses the context
grounded = (
    "Compound interest on a savings account works by calculating interest "
    "on both the principal and previously accumulated interest, causing "
    "the balance to grow exponentially over time."
)

# A hallucinated response — fabricated mechanism
hallucinated = (
    "Compound interest works by applying a fixed annual rate only to the "
    "original deposit, with accumulated interest stored in a separate "
    "non-yielding reserve account managed by the Federal Reserve."
)

sgi_grounded = compute_sgi(question=question, context=context, response=grounded)
sgi_hallucinated = compute_sgi(question=question, context=context, response=hallucinated)

print("=" * 60)
print("SGI — Context-Based Verification")
print("=" * 60)
print(f"\n✅ Grounded response:")
print(f"   SGI = {sgi_grounded.value:.3f}  (normalized: {sgi_grounded.normalized:.3f})")
print(f"   {sgi_grounded.explanation}")
print(f"\n❌ Hallucinated response:")
print(f"   SGI = {sgi_hallucinated.value:.3f}  (normalized: {sgi_hallucinated.normalized:.3f})")
print(f"   {sgi_hallucinated.explanation}")

## 3. DGI — Context-Free Verification

When there's no source context (e.g., open-ended generation), DGI checks whether the **direction** of semantic movement from question to response matches the pattern of verified grounded responses.

$$\text{DGI} = \hat{\delta}^\top \hat{\mu}$$

where $\hat{\delta}$ is the unit displacement and $\hat{\mu}$ is a reference direction learned from calibration pairs.

- DGI > 0.30 → aligns with grounded patterns (pass)
- DGI < 0.30 → diverges from grounded direction (flagged)

In [ ]:
from factlens import compute_dgi

question = "What causes the Northern Lights?"

# Grounded answer
grounded = (
    "The Northern Lights, or aurora borealis, are caused by charged particles "
    "from the sun interacting with Earth's magnetic field. These particles "
    "collide with atmospheric gases, producing colorful light displays."
)

# Hallucinated answer — plausible-sounding but fabricated
hallucinated = (
    "The Northern Lights are caused by underground magma chambers releasing "
    "ionized xenon gas through volcanic micro-fissures in the Arctic "
    "tectonic plate, a process known as subterranean photoemission."
)

dgi_grounded = compute_dgi(question=question, response=grounded)
dgi_hallucinated = compute_dgi(question=question, response=hallucinated)

print("=" * 60)
print("DGI — Context-Free Verification")
print("=" * 60)
print("\ Grounded response:")
print("   DGI = {dgi_grounded.value:.3f}  (normalized: {dgi_grounded.normalized:.3f})")
print("   {dgi_grounded.explanation}")
print("\ Hallucinated response:")
print("   DGI = {dgi_hallucinated.value:.3f}  (normalized: {dgi_hallucinated.normalized:.3f})")
print("   {dgi_hallucinated.explanation}")

## 4. The `evaluate()` Function — Auto-Select

The high-level `evaluate()` function picks SGI or DGI automatically based on whether you provide context.

In [ ]:
from factlens import evaluate

# With context → uses SGI
score_sgi = evaluate(
    question="What is a 401(k) employer match?",
    response=(
        "An employer match is when a company contributes to your 401(k) "
        "based on your own contributions, commonly matching 50% to 100% "
        "of contributions up to a certain percentage of salary."
    ),
    context=(
        "A 401(k) employer match is a benefit where the employer deposits "
        "money into an employee's retirement account proportional to the "
        "employee's own contributions, typically up to a percentage of salary."
    ),
)

# Without context → uses DGI
score_dgi = evaluate(
    question="How do vaccines work?",
    response=(
        "Vaccines introduce a weakened or inactivated form of a pathogen "
        "to train the immune system to recognize and fight the actual infection."
    ),
)

print("evaluate() auto-selects the right method:")
print(f"\n  With context → method={score_sgi.method}, value={score_sgi.value:.3f}, flagged={score_sgi.flagged}")
print(f"  No context   → method={score_dgi.method}, value={score_dgi.value:.3f}, flagged={score_dgi.flagged}")

## 5. Batch Evaluation — Score a Dataset

Let's run factlens on a set of grounded vs. hallucinated responses across domains and visualize the results.

In [ ]:
test_cases = [
    # (domain, question, context, response, is_grounded)
    (
        "Finance",
        "What is the standard deductible for homeowner's insurance?",
        "The standard deductible for homeowner's insurance typically ranges from $500 to $2,000, depending on the policy and insurer.",
        "Homeowner's insurance deductibles typically range from $500 to $2,000, varying by policy.",
        True,
    ),
    (
        "Finance",
        "What is the standard deductible for homeowner's insurance?",
        "The standard deductible for homeowner's insurance typically ranges from $500 to $2,000, depending on the policy and insurer.",
        "The standard deductible is always exactly $750, as mandated by the Federal Insurance Regulation Act of 1997.",
        False,
    ),
    (
        "Medical",
        "What are the symptoms of Type 2 diabetes?",
        "Common symptoms include increased thirst, frequent urination, unexplained weight loss, fatigue, blurred vision, and slow-healing wounds.",
        "Symptoms include increased thirst, frequent urination, fatigue, blurred vision, and slow-healing wounds.",
        True,
    ),
    (
        "Medical",
        "What are the symptoms of Type 2 diabetes?",
        "Common symptoms include increased thirst, frequent urination, unexplained weight loss, fatigue, blurred vision, and slow-healing wounds.",
        "The primary symptom is intermittent tingling in the left hand, caused by Langerhans cells in the liver failing to produce sufficient glucagon.",
        False,
    ),
    (
        "Science",
        "What is photosynthesis?",
        "Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into glucose and oxygen, using chlorophyll.",
        "Photosynthesis converts sunlight, water, and CO2 into glucose and oxygen using chlorophyll in plant leaves.",
        True,
    ),
    (
        "Science",
        "What is photosynthesis?",
        "Photosynthesis is the process by which plants convert sunlight, water, and carbon dioxide into glucose and oxygen, using chlorophyll.",
        "Photosynthesis is the process by which plants absorb nitrogen from the air through their root system and convert it to chlorophyll using the Kramer-Holst cycle.",
        False,
    ),
    (
        "Legal",
        "What is the statute of limitations for personal injury claims?",
        "The statute of limitations for personal injury claims varies by state, typically ranging from 1 to 6 years from the date of injury.",
        "Personal injury statutes of limitations vary by state, generally ranging from 1 to 6 years.",
        True,
    ),
    (
        "Legal",
        "What is the statute of limitations for personal injury claims?",
        "The statute of limitations for personal injury claims varies by state, typically ranging from 1 to 6 years from the date of injury.",
        "Under the Universal Tort Reform Act of 2019, all personal injury claims must be filed within exactly 90 days, regardless of jurisdiction.",
        False,
    ),
]

print(f"Running SGI on {len(test_cases)} test cases...\n")

In [ ]:
results = []
for domain, q, ctx, resp, is_grounded in test_cases:
    sgi = compute_sgi(question=q, context=ctx, response=resp)
    results.append({
        "domain": domain,
        "grounded": is_grounded,
        "sgi_raw": sgi.value,
        "sgi_norm": sgi.normalized,
        "flagged": sgi.flagged,
    })

# Display as a table
print(f"{'Domain':<10} {'Ground Truth':<14} {'SGI':>6} {'Norm':>6} {'Flagged':>8} {'Verdict':>10}")
print("─" * 62)
for r in results:
    truth = "✅ Grounded" if r["grounded"] else "❌ Halluc."
    verdict = "⚠️  REVIEW" if r["flagged"] else "✓ PASS"
    print(f"{r['domain']:<10} {truth:<14} {r['sgi_raw']:>6.3f} {r['sgi_norm']:>6.3f} {str(r['flagged']):>8} {verdict:>10}")

# Summary
grounded_flagged = sum(1 for r in results if r["grounded"] and r["flagged"])
halluc_flagged = sum(1 for r in results if not r["grounded"] and r["flagged"])
total_grounded = sum(1 for r in results if r["grounded"])
total_halluc = sum(1 for r in results if not r["grounded"])

print(f"\n📊 Summary:")
print(f"   Grounded correctly passed:    {total_grounded - grounded_flagged}/{total_grounded}")
print(f"   Hallucinations caught:        {halluc_flagged}/{total_halluc}")

## 6. Visualize — SGI Score Distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

grounded_scores = [r["sgi_raw"] for r in results if r["grounded"]]
halluc_scores = [r["sgi_raw"] for r in results if not r["grounded"]]
domains_g = [r["domain"] for r in results if r["grounded"]]
domains_h = [r["domain"] for r in results if not r["grounded"]]

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(grounded_scores))
width = 0.35

bars_g = ax.bar(x - width/2, grounded_scores, width, label="Grounded", color="#2196F3", alpha=0.85)
bars_h = ax.bar(x + width/2, halluc_scores, width, label="Hallucinated", color="#F44336", alpha=0.85)

# Threshold lines
ax.axhline(y=1.20, color="green", linestyle="--", alpha=0.6, label="Strong Pass (1.20)")
ax.axhline(y=0.95, color="orange", linestyle="--", alpha=0.6, label="Review Threshold (0.95)")

ax.set_xlabel("Domain")
ax.set_ylabel("SGI Score")
ax.set_title("SGI: Grounded vs. Hallucinated Responses")
ax.set_xticks(x)
ax.set_xticklabels(domains_g)
ax.legend(loc="upper right")
ax.set_ylim(0, max(grounded_scores + halluc_scores) * 1.2)

plt.tight_layout()
plt.show()

## 7. Under the Hood — The Geometry

Let's peek at what factlens actually computes. All operations are deterministic — same input always produces the same score.

In [ ]:
from factlens._internal.embeddings import encode_texts
from factlens._internal.geometry import euclidean_distance
import numpy as np

question = "What causes the Northern Lights?"
context = (
    "The Northern Lights are caused by charged particles from the sun "
    "interacting with Earth's magnetic field and colliding with atmospheric gases."
)
grounded = (
    "Aurora borealis occurs when solar wind particles interact with "
    "Earth's magnetosphere and collide with atmospheric molecules."
)
hallucinated = (
    "The Northern Lights are caused by underground magma chambers "
    "releasing ionized xenon gas through Arctic tectonic micro-fissures."
)

# Embed all four texts
embeddings = encode_texts([question, context, grounded, hallucinated])
q_emb, ctx_emb, g_emb, h_emb = embeddings

print(f"Embedding dimension: {q_emb.shape[0]}")
print(f"\nEuclidean distances:")
print(f"  question  ↔ context:       {euclidean_distance(q_emb, ctx_emb):.4f}")
print(f"  grounded  ↔ question:      {euclidean_distance(g_emb, q_emb):.4f}")
print(f"  grounded  ↔ context:       {euclidean_distance(g_emb, ctx_emb):.4f}")
print(f"  halluc.   ↔ question:      {euclidean_distance(h_emb, q_emb):.4f}")
print(f"  halluc.   ↔ context:       {euclidean_distance(h_emb, ctx_emb):.4f}")

sgi_g = euclidean_distance(g_emb, q_emb) / euclidean_distance(g_emb, ctx_emb)
sgi_h = euclidean_distance(h_emb, q_emb) / euclidean_distance(h_emb, ctx_emb)

print(f"\nManual SGI computation:")
print(f"  Grounded:     dist(r,q)/dist(r,ctx) = {sgi_g:.3f}  {'✅ > 1' if sgi_g > 1 else '⚠️  < 1'}")
print(f"  Hallucinated: dist(r,q)/dist(r,ctx) = {sgi_h:.3f}  {'✅ > 1' if sgi_h > 1 else '⚠️  < 1'}")
print(f"\n→ The grounded response is closer to the context (SGI > 1).")
print(f"→ The hallucination is closer to the question (SGI < 1).")

## 8. Try Your Own Examples

Replace the strings below with your own question, context, and response to see how factlens scores them.

In [ ]:
from factlens import evaluate

# ─── Edit these ───────────────────────────────────────────────
my_question = "What is the speed of light?"
my_context  = "The speed of light in a vacuum is approximately 299,792,458 meters per second."  # set to None for DGI
my_response = "Light travels at about 300,000 km/s in a vacuum."
# ──────────────────────────────────────────────────────────────

score = evaluate(
    question=my_question,
    response=my_response,
    context=my_context,
)

print(f"Method:      {score.method.upper()}")
print(f"Raw score:   {score.value:.3f}")
print(f"Normalized:  {score.normalized:.3f}")
print(f"Flagged:     {score.flagged}")
print(f"Explanation: {score.explanation}")

## What's Next?

- **Domain calibration**: Train DGI on your own verified pairs for AUROC 0.90–0.99 → [Guide](https://docs.factlens.dev/guides/domain-calibration/)
- **RAG verification**: Plug factlens into your retrieval pipeline → [Guide](https://docs.factlens.dev/guides/rag-verification/)
- **Framework integrations**: LangChain, CrewAI, Semantic Kernel, AutoGen → [Docs](https://docs.factlens.dev/integrations/langchain/)
- **EU AI Act compliance**: Use factlens for Article 15 risk management → [Guide](https://docs.factlens.dev/guides/eu-ai-act/)

```bash
pip install factlens
```

⭐ Star the repo if this was useful: [github.com/factlens/factlens](https://github.com/factlens/factlens)